In [ ]:
from IPython.display import Markdown, HTML

from tvbo import Dynamics as DynamicalSystem
from tvbo import SimulationExperiment

system = DynamicalSystem(
    state_variables=[
        {
            "name": "x",
            "equation": {"rhs": "v"},
            "initial_value": 2.0,
        },
        {
            "name": "v",
            "equation": {"rhs": "-w**2 * x"},
            "initial_value": 0.0,
        },
    ],
    parameters=[
        {"name": "w", "value": 0.01},
    ],
)

Markdown(system.render('markdown'))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

for init_x in [0.5, 1.0, 1.5, 2.0]:
    system.state_variables['x']['initial_value'] = init_x
    system.plot(kind='phase', ax=ax, cmap='grey_r')

system.plot(kind='vectorfield', ax=ax, alpha=0.3)
ax.set_xlabel("x (position)")
ax.set_ylabel("v (velocity)")

In [ ]:
system.state_variables['x']['initial_value'] = 2.0
result = SimulationExperiment(dynamics=system).run()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Extract time series via xarray named indexing
x_ts = np.asarray(result.data.sel({"variable": "x"}))
t = np.linspace(0, 1, len(x_ts))

# Spring drawing helper
def draw_spring(x0, x1, n_coils=8, amplitude=0.15):
    n_pts = n_coils * 20 + 2
    xs = np.linspace(x0, x1, n_pts)
    ys = np.zeros(n_pts)
    ys[1:-1] = amplitude * np.sin(np.linspace(0, n_coils * 2 * np.pi, n_pts - 2))
    return xs, ys

wall_x = -3.0
mass_w, mass_h = 0.3, 0.3

fig, (ax_spring, ax_ts) = plt.subplots(2, 1, figsize=(7, 5),
                                        gridspec_kw={'height_ratios': [2, 1]})
fig.tight_layout(pad=2)

ax_spring.set_xlim(wall_x - 0.2, 3.5)
ax_spring.set_ylim(-0.6, 0.6)
ax_spring.set_aspect('equal')
ax_spring.axis('off')
ax_spring.fill_betweenx([-0.5, 0.5], wall_x - 0.25, wall_x, color='gray')
ax_spring.axvline(wall_x, color='gray', lw=2)
ax_spring.axvline(0, color='red', lw=0.8, ls='--', alpha=0.5, label='equilibrium')
ax_spring.legend(loc='upper right', fontsize=8)

spring_line, = ax_spring.plot([], [], 'k-', lw=1.5)
mass_rect = plt.Rectangle((0, -mass_h / 2), mass_w, mass_h, fc='steelblue', ec='k', zorder=5)
ax_spring.add_patch(mass_rect)

ax_ts.set_xlim(0, 1)
ax_ts.set_ylim(x_ts.min() * 1.1, x_ts.max() * 1.1)
ax_ts.set_xlabel("time (normalised)")
ax_ts.set_ylabel("x")
ax_ts.plot(t, x_ts, color='lightgray', lw=1)
ts_dot, = ax_ts.plot([], [], 'o', color='steelblue', ms=5)
ts_line, = ax_ts.plot([], [], color='steelblue', lw=1.5)

def init():
    spring_line.set_data([], [])
    mass_rect.set_xy((-mass_w / 2, -mass_h / 2))
    ts_dot.set_data([], [])
    ts_line.set_data([], [])
    return spring_line, mass_rect, ts_dot, ts_line

stride = max(1, len(x_ts) // 200)

def update(frame):
    i = frame * stride
    xi = float(x_ts[i])
    sx, sy = draw_spring(wall_x, xi - mass_w / 2)
    spring_line.set_data(sx, sy)
    mass_rect.set_xy((xi - mass_w / 2, -mass_h / 2))
    ts_dot.set_data([t[i]], [xi])
    ts_line.set_data(t[:i+1], x_ts[:i+1])
    return spring_line, mass_rect, ts_dot, ts_line

ani = animation.FuncAnimation(fig, update, frames=len(x_ts) // stride,
                               init_func=init, blit=True, interval=30)
plt.close()
ani

HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# --- 3 initial conditions, same parameters ---
init_xs = [1.0, 2.0, 3.0]
colors  = ['#4e79a7', '#f28e2b', '#59a14f']

results_multi = []
for x0 in init_xs:
    system.state_variables['x']['initial_value'] = x0
    system.state_variables['v']['initial_value'] = 0.0
    results_multi.append(SimulationExperiment(dynamics=system).run())

ts_arrays = [np.asarray(r.data.sel({"variable": "x"})).squeeze() for r in results_multi]
n_pts = min(len(a) for a in ts_arrays)
ts_arrays = [a[:n_pts] for a in ts_arrays]
t = np.linspace(0, 1, n_pts)

def draw_spring_v(y0, y1, n_coils=8, amplitude=0.08):
    n = n_coils * 20 + 2
    ys = np.linspace(y0, y1, n)
    xs = np.zeros(n)
    xs[1:-1] = amplitude * np.sin(np.linspace(0, n_coils * 2 * np.pi, n - 2))
    return xs, ys

ceiling_y = 3.5
mass_h, mass_w = 0.4, 0.4
# Same ylim for all panels so equilibrium (y=0) is at the same visual height
amp_max   = max(init_xs)
y_lo, y_hi = -(amp_max + mass_h + 0.3), ceiling_y + 0.4

fig, axes = plt.subplot_mosaic(
    [['a', 'b', 'c'], ['t', 't', 't']],
    figsize=(9, 6),
    gridspec_kw={'height_ratios': [2, 1]},
)
fig.tight_layout(pad=2.5)

spring_lines, mass_rects = [], []
for ax_key, x0, col in zip(['a', 'b', 'c'], init_xs, colors):
    ax = axes[ax_key]
    ax.set_xlim(-1, 1)
    ax.set_ylim(y_lo, y_hi)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'$x_0 = {x0}$', fontsize=9, color=col)
    ax.fill_between([-0.6, 0.6], ceiling_y, ceiling_y + 0.25, color='gray')
    ax.axhline(ceiling_y, color='gray', lw=2)
    ax.axhline(0, color='red', lw=0.8, ls='--', alpha=0.4)
    sl, = ax.plot([], [], 'k-', lw=1.5)
    mr = plt.Rectangle((-mass_w / 2, -mass_h / 2), mass_w, mass_h,
                        fc=col, ec='k', zorder=5, alpha=0.85)
    ax.add_patch(mr)
    spring_lines.append(sl)
    mass_rects.append(mr)

ax_t = axes['t']
ax_t.set_xlim(0, 1)
ymin = min(a.min() for a in ts_arrays) * 1.15
ymax = max(a.max() for a in ts_arrays) * 1.15
ax_t.set_ylim(ymin, ymax)
ax_t.set_xlabel("time (normalised)")
ax_t.set_ylabel("x")
ax_t.axhline(0, color='red', lw=0.8, ls='--', alpha=0.4)
ts_anim_lines, ts_dots = [], []
for arr, col, x0 in zip(ts_arrays, colors, init_xs):
    ax_t.plot(t, arr, color=col, lw=0.8, alpha=0.25)
    al, = ax_t.plot([], [], color=col, lw=1.5, label=f'$x_0={x0}$')
    td, = ax_t.plot([], [], 'o', color=col, ms=5)
    ts_anim_lines.append(al)
    ts_dots.append(td)
ax_t.legend(fontsize=8, loc='upper right')

stride = max(1, n_pts // 200)

def init():
    for sl, mr in zip(spring_lines, mass_rects):
        sl.set_data([], [])
        mr.set_xy((-mass_w / 2, -mass_h / 2))
    for al, td in zip(ts_anim_lines, ts_dots):
        al.set_data([], [])
        td.set_data([], [])
    return (*spring_lines, *mass_rects, *ts_anim_lines, *ts_dots)

def update(frame):
    i = frame * stride
    artists = []
    for sl, mr, arr in zip(spring_lines, mass_rects, ts_arrays):
        xi = float(arr[i])
        sl.set_data(*draw_spring_v(ceiling_y, xi + mass_h / 2))
        mr.set_xy((-mass_w / 2, xi - mass_h / 2))
        artists += [sl, mr]
    for al, td, arr in zip(ts_anim_lines, ts_dots, ts_arrays):
        xi = float(arr[i])
        al.set_data(t[:i+1], arr[:i+1])
        td.set_data([t[i]], [xi])
        artists += [al, td]
    return artists

ani = animation.FuncAnimation(fig, update, frames=n_pts // stride,
                               init_func=init, blit=True, interval=30)
plt.close()
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# --- Same initial condition, 3 different frequencies (w) ---
w_values = [0.005, 0.01, 0.02]
colors   = ['#4e79a7', '#f28e2b', '#59a14f']
x0_fixed = 2.0

results_w = []
for w in w_values:
    system.state_variables['x']['initial_value'] = x0_fixed
    system.state_variables['v']['initial_value'] = 0.0
    system.parameters['w']['value'] = w
    results_w.append(SimulationExperiment(dynamics=system).run())

# Restore default
system.parameters['w']['value'] = 0.01

ts_w = [np.asarray(r.data.sel({"variable": "x"})).squeeze() for r in results_w]
n_pts = min(len(a) for a in ts_w)
ts_w  = [a[:n_pts] for a in ts_w]
t     = np.linspace(0, 1, n_pts)

def draw_spring_v(y0, y1, n_coils=8, amplitude=0.08):
    n = n_coils * 20 + 2
    ys = np.linspace(y0, y1, n)
    xs = np.zeros(n)
    xs[1:-1] = amplitude * np.sin(np.linspace(0, n_coils * 2 * np.pi, n - 2))
    return xs, ys

ceiling_y = 3.5
mass_h, mass_w = 0.4, 0.4
y_lo, y_hi = -(x0_fixed + mass_h + 0.3), ceiling_y + 0.4

fig, axes = plt.subplot_mosaic(
    [['a', 'b', 'c'], ['t', 't', 't']],
    figsize=(9, 6),
    gridspec_kw={'height_ratios': [2, 1]},
)
fig.tight_layout(pad=2.5)

spring_lines, mass_rects = [], []
for ax_key, w, col in zip(['a', 'b', 'c'], w_values, colors):
    ax = axes[ax_key]
    ax.set_xlim(-1, 1)
    ax.set_ylim(y_lo, y_hi)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'$\\omega = {w}$', fontsize=9, color=col)
    ax.fill_between([-0.6, 0.6], ceiling_y, ceiling_y + 0.25, color='gray')
    ax.axhline(ceiling_y, color='gray', lw=2)
    ax.axhline(0, color='red', lw=0.8, ls='--', alpha=0.4)
    sl, = ax.plot([], [], 'k-', lw=1.5)
    mr = plt.Rectangle((-mass_w / 2, -mass_h / 2), mass_w, mass_h,
                        fc=col, ec='k', zorder=5, alpha=0.85)
    ax.add_patch(mr)
    spring_lines.append(sl)
    mass_rects.append(mr)

ax_t = axes['t']
ax_t.set_xlim(0, 1)
ymin = min(a.min() for a in ts_w) * 1.15
ymax = max(a.max() for a in ts_w) * 1.15
ax_t.set_ylim(ymin, ymax)
ax_t.set_xlabel("time (normalised)")
ax_t.set_ylabel("x")
ax_t.axhline(0, color='red', lw=0.8, ls='--', alpha=0.4)
ts_anim_lines, ts_dots = [], []
for arr, col, w in zip(ts_w, colors, w_values):
    ax_t.plot(t, arr, color=col, lw=0.8, alpha=0.25)
    al, = ax_t.plot([], [], color=col, lw=1.5, label=f'$\\omega={w}$')
    td, = ax_t.plot([], [], 'o', color=col, ms=5)
    ts_anim_lines.append(al)
    ts_dots.append(td)
ax_t.legend(fontsize=8, loc='upper right')

stride = max(1, n_pts // 200)

def init2():
    for sl, mr in zip(spring_lines, mass_rects):
        sl.set_data([], [])
        mr.set_xy((-mass_w / 2, -mass_h / 2))
    for al, td in zip(ts_anim_lines, ts_dots):
        al.set_data([], [])
        td.set_data([], [])
    return (*spring_lines, *mass_rects, *ts_anim_lines, *ts_dots)

def update2(frame):
    i = frame * stride
    artists = []
    for sl, mr, arr in zip(spring_lines, mass_rects, ts_w):
        xi = float(arr[i])
        sl.set_data(*draw_spring_v(ceiling_y, xi + mass_h / 2))
        mr.set_xy((-mass_w / 2, xi - mass_h / 2))
        artists += [sl, mr]
    for al, td, arr in zip(ts_anim_lines, ts_dots, ts_w):
        xi = float(arr[i])
        al.set_data(t[:i+1], arr[:i+1])
        td.set_data([t[i]], [xi])
        artists += [al, td]
    return artists

ani2 = animation.FuncAnimation(fig, update2, frames=n_pts // stride,
                                init_func=init2, blit=True, interval=30)
plt.close()
HTML(ani2.to_jshtml())